# Stage 1: Text Preprocessing with spaCy

This notebook explores how spaCy cleans raw review text before passing it to RoBERTa for sentiment classification.

Pipeline:
- Lowercase text
- Tokenize
- Lemmatize
- Remove stop words and punctuation

In [1]:
import spacy

# Load the English language model
nlp = spacy.load('en_core_web_sm')

print('spaCy loaded successfully')
print('Pipeline components:', nlp.pipe_names)

spaCy loaded successfully
Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [2]:
# Process a sample review
review = "This product is ABSOLUTELY Amazing!!! I've never seen anything like it."

# nlp() runs the full spaCy pipeline on your text
doc = nlp(review)

# Look at what spaCy sees for each token
print(f"{'TOKEN':<15} {'LEMMA':<15} {'POS':<10} {'IS_STOP':<10} {'IS_PUNCT'}")
print("-" * 60)

for token in doc:
    print(f"{token.text:<15} {token.lemma_:<15} {token.pos_:<10} {str(token.is_stop):<10} {token.is_punct}")

TOKEN           LEMMA           POS        IS_STOP    IS_PUNCT
------------------------------------------------------------
This            this            DET        True       False
product         product         NOUN       False      False
is              be              AUX        True       False
ABSOLUTELY      ABSOLUTELY      PROPN      False      False
Amazing         Amazing         PROPN      False      False
!               !               PUNCT      False      True
!               !               PUNCT      False      True
!               !               PUNCT      False      True
I               I               PRON       True       False
've             have            AUX        True       False
never           never           ADV        True       False
seen            see             VERB       False      False
anything        anything        PRON       True       False
like            like            ADP        False      False
it              it              PRON   

In [3]:
def preprocess_text(text):
    #Lowercase the text before giving it to spaCy
    # This fixes the ABSOLUTELY and Amazing tagging problem we just saw
    text = text.lower()
   
    doc = nlp(text)
    
    # build a list of clean tokens
    tokens = [
        token.lemma_          # use lemma (root form), not original word
        for token in doc
        if not token.is_stop  # remove stop words
        and not token.is_punct # remove punctuation
        and not token.is_space # remove whitespace tokens
    ]
    
    # RoBERTa expects a string, not a list
    return " ".join(tokens)

## Testing the Preprocessing Function

In [4]:
# Test with the same review we analyzed in Step 2
review = "This product is ABSOLUTELY Amazing!!! I've never seen anything like it."

cleaned = preprocess_text(review)

print("Original:", review)
print("Cleaned: ", cleaned)


Original: This product is ABSOLUTELY Amazing!!! I've never seen anything like it.
Cleaned:  product absolutely amazing see like


## Improved Preprocessing — Preserving Sentiment Words`

In [5]:
# These words are marked as stop words by spaCy
# but they carry strong sentiment meaning
# We define them manually so our function keeps them
SENTIMENT_STOP_EXCEPTIONS = {
    "never", "no", "not", "neither", "nor",
    "nobody", "nothing", "nowhere", "hardly",
    "barely", "scarcely"
}

def preprocess_text(text):
    text = text.lower()
    
    doc = nlp(text)
    
    tokens = [
        token.lemma_
        for token in doc
        if (not token.is_stop or token.lemma_ in SENTIMENT_STOP_EXCEPTIONS)
        and not token.is_punct
        and not token.is_space
    ]
    
    return " ".join(tokens)

# Test it
review = "This product is ABSOLUTELY Amazing!!! I've never seen anything like it."
cleaned = preprocess_text(review)

print("Original:", review)
print("Cleaned: ", cleaned)

Original: This product is ABSOLUTELY Amazing!!! I've never seen anything like it.
Cleaned:  product absolutely amazing never see like


## Testing With Multiple Review Types

In [6]:
test_reviews = [
    "This product is ABSOLUTELY Amazing!!! I've never seen anything like it.",
    "Terrible quality. The item broke after 2 days and customer service did nothing.",
    "It's okay I guess. Not great, not terrible. Does what it says.",
    "I can't believe how bad this is!!! Complete waste of money!!!",
    "Best purchase I've ever made. Highly recommend to everyone!"
]

print(f"{'ORIGINAL':<60} {'CLEANED'}")
print("-" * 100)

for review in test_reviews:
    cleaned = preprocess_text(review)
    print(f"{review:<60} {cleaned}")
    print()

ORIGINAL                                                     CLEANED
----------------------------------------------------------------------------------------------------
This product is ABSOLUTELY Amazing!!! I've never seen anything like it. product absolutely amazing never see like

Terrible quality. The item broke after 2 days and customer service did nothing. terrible quality item break 2 day customer service nothing

It's okay I guess. Not great, not terrible. Does what it says. okay guess not great not terrible say

I can't believe how bad this is!!! Complete waste of money!!! not believe bad complete waste money

Best purchase I've ever made. Highly recommend to everyone!  good purchase highly recommend



## Testing With Real Amazon Reviews

In [7]:
import pandas as pd

df = pd.read_csv('/kaggle/input/datasets/aliibtissam/sentiment-analysis-data/Reviews.csv')

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print(df['Text'].head())

Shape: (568454, 10)
Columns: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']

0    I have bought several of the Vitality canned d...
1    Product arrived labeled as Jumbo Salted Peanut...
2    This is a confection that has been around a fe...
3    If you are looking for the secret ingredient i...
4    Great taffy at a great price.  There was a wid...
Name: Text, dtype: object


## Exploring the Score Column

In [8]:
print('Score distribution:')
print(df['Score'].value_counts().sort_index())
print()
print('Null values in Text column:', df['Text'].isnull().sum())
print('Null values in Score column:', df['Score'].isnull().sum())

Score distribution:
Score
1     52268
2     29769
3     42640
4     80655
5    363122
Name: count, dtype: int64

Null values in Text column: 0
Null values in Score column: 0


In [9]:
def map_sentiment(score):
    if score <= 2:
        return 'negative'
    elif score == 3:
        return 'neutral'
    else:
        return 'positive'

df['sentiment'] = df['Score'].apply(map_sentiment)

print('Sentiment distribution:')
print(df['sentiment'].value_counts())
print()
print(df[['Text', 'Score', 'sentiment']].head(10))

Sentiment distribution:
sentiment
positive    443777
negative     82037
neutral      42640
Name: count, dtype: int64

                                                Text  Score sentiment
0  I have bought several of the Vitality canned d...      5  positive
1  Product arrived labeled as Jumbo Salted Peanut...      1  negative
2  This is a confection that has been around a fe...      4  positive
3  If you are looking for the secret ingredient i...      2  negative
4  Great taffy at a great price.  There was a wid...      5  positive
5  I got a wild hair for taffy and ordered this f...      4  positive
6  This saltwater taffy had great flavors and was...      5  positive
7  This taffy is so good.  It is very soft and ch...      5  positive
8  Right now I'm mostly just sprouting this so my...      5  positive
9  This is a very healthy dog food. Good for thei...      5  positive


## Creating a Balanced Sample for Exploration

In [10]:
# Take 5000 from each class for a balanced sample
sample_size = 5000

df_negative = df[df['sentiment'] == 'negative'].sample(sample_size, random_state=42)
df_neutral = df[df['sentiment'] == 'neutral'].sample(sample_size, random_state=42)
df_positive = df[df['sentiment'] == 'positive'].sample(sample_size, random_state=42)

# Combine and shuffle
df_sample = pd.concat([df_negative, df_neutral, df_positive])
df_sample = df_sample.sample(frac=1, random_state=42).reset_index(drop=True)

print('Sample shape:', df_sample.shape)
print()
print('Sample sentiment distribution:')
print(df_sample['sentiment'].value_counts())

Sample shape: (15000, 11)

Sample sentiment distribution:
sentiment
positive    5000
neutral     5000
negative    5000
Name: count, dtype: int64


In [11]:
df_sample.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,sentiment
0,461431,B00292BMN6,A2MVOKR2U9MAV8,L.K.,0,0,4,1347062400,My dogs love this treat,My 6 month old puppies and adult dogs love thi...,positive
1,244729,B001FA1E0O,AHPGZ6UOX9LPR,C. Ulickey,1,1,3,1265760000,A mite disappointed,I originally purchased the Mauna Loa chocolate...,neutral
2,18426,B000NVD9DG,A1ROC7YRFP15BW,Deborah Davis,0,0,5,1321315200,Yum Yum,My Maltese loves these treats and has never ha...,positive
3,399058,B0038B3AGO,A3FVQFRT5PMKD0,S. Field,0,9,1,1298937600,nonsoluable,"i have no doubt the contents are valuable, but...",negative
4,289243,B000I9R0AG,A2HICFJKDEV17R,sweetgirl,0,0,3,1336608000,way to expensive,these mints were good but not worth the price ...,neutral


## Applying Preprocessing to Sample Dataset

In [12]:
# This will take a minute to run on 15000 rows
df_sample['cleaned_text'] = df_sample['Text'].apply(preprocess_text)

print('Done.')
print()
print(df_sample[['Text', 'cleaned_text', 'sentiment']].head(5))

Done.

                                                Text  \
0  My 6 month old puppies and adult dogs love thi...   
1  I originally purchased the Mauna Loa chocolate...   
2  My Maltese loves these treats and has never ha...   
3  i have no doubt the contents are valuable, but...   
4  these mints were good but not worth the price ...   

                                        cleaned_text sentiment  
0  6 month old puppy adult dog love treat long la...  positive  
1  originally purchase mauna loa chocolate cover ...   neutral  
2  maltese love treat never adverse reaction hard...  positive  
3  no doubt content valuable not mix liquid good ...  negative  
4  mint good not worth price pay plus ridiculous ...   neutral  


## Preprocessing Quality Check

In [13]:
    # Check token length distribution
df_sample['token_count'] = df_sample['cleaned_text'].apply(lambda x: len(x.split()))

print('Token count statistics:')
print(df_sample['token_count'].describe())
print()

# Check for empty reviews after cleaning
empty_reviews = df_sample[df_sample['cleaned_text'].str.strip() == '']
print('Empty reviews after cleaning:', len(empty_reviews))
print()

# Show shortest reviews to check quality
print('5 shortest cleaned reviews:')
print(df_sample.nsmallest(5, 'token_count')[['Text', 'cleaned_text', 'token_count']])

Token count statistics:
count    15000.000000
mean        41.482467
std         43.275698
min          4.000000
25%         17.000000
50%         29.000000
75%         51.000000
max       1889.000000
Name: token_count, dtype: float64

Empty reviews after cleaning: 0

5 shortest cleaned reviews:
                                                   Text  \
2247  Just as I remembered It when I bought one in T...   
3525  This is a great tasting, good for you, snack t...   
9425                  Same price as Dr. Foster & Smith.   
124   tastes good, but a little on the sweet side.  ...   
924   I KEEP THIS AT WORK AND I MIX IN MY SMOOTHIES ...   

                     cleaned_text  token_count  
2247   remember buy caymen island            4  
3525     great tasting good snack            4  
9425        price dr foster smith            4  
124   taste good little sweet not            5  
924   work mix smoothy home- love            5  


## Saving Cleaned Sample for Next Notebook

In [14]:
# Keep only the columns we need going forward
df_save = df_sample[['Text', 'cleaned_text', 'sentiment', 'Score']].copy()

df_save.to_csv('amazon_reviews_cleaned_sample.csv', index=False)

print('Saved successfully.')
print('Shape:', df_save.shape)
print('Columns:', df_save.columns.tolist())

Saved successfully.
Shape: (15000, 4)
Columns: ['Text', 'cleaned_text', 'sentiment', 'Score']
